# Model and behavioral-rubric audit — Appendices B and E
Rebuilds the prior sensitivity and every Appendix E rubric diagnostic. Transformations live in `aerobat.analysis`; this notebook only selects paths, runs the analyses, and displays their outputs. Rubric refits take several minutes.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'run_analysis': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from aerobat.analysis.sensitivity import prior_sensitivity, prior_sensitivity_summary
from aerobat.analysis.rubric import (
    EMBEDDING_MODEL, generate_embedding_cache, internal_consistency,
    load_embedding_cache, load_rubric_score_cells, null_scores,
    rubric_criteria, rubric_sensitivity, semantic_specificity,
    sensitivity_summary, write_appendix_e_outputs,
)
from aerobat.analysis.tables import (
    prior_sensitivity_table, write_prior_sensitivity_latex,
    rubric_semantics_table, write_rubric_consistency_latex,
    write_rubric_semantics_latex, write_rubric_sensitivity_latex,
)
RESULTS = ROOT / 'results' / 'GPT-5-mini'
OUTPUT = ROOT / 'run_analysis' / 'outputs'
EMBEDDING_CACHE = ROOT / 'run_analysis' / 'inputs' / 'rubric_embeddings_text_embedding_3_large.json'
OUTPUT.mkdir(parents=True, exist_ok=True)

## Appendix B: prior scale
The hypothesis count is computed from the current results; no manuscript count is typed into the notebook.

In [ ]:
prior = prior_sensitivity(RESULTS)
prior_summary = prior_sensitivity_summary(prior)
prior_table = prior_sensitivity_table(prior)
prior.to_csv(OUTPUT / 'prior_sensitivity.csv', index=False)
prior_table.to_csv(OUTPUT / 'prior_sensitivity_summary.csv', index=False)
write_prior_sensitivity_latex(prior_table, OUTPUT / 'tabular_stat_sensitivity.tex')
display(prior_summary)
display(prior_table)
print('Hypotheses:', prior[['behavior_name', 'axis_slug']].drop_duplicates().shape[0])

## Appendix E: rubric measurements
Explicit nulls retain their Stage 4 meaning. The reliability audit treats evidence classes as items and residualizes matched-block and hypothesized-cause level effects.

In [ ]:
criteria = rubric_criteria(RESULTS)
score_cells = load_rubric_score_cells(RESULTS)
consistency = internal_consistency(score_cells)
null_diagnostics = null_scores(score_cells)
display(consistency.summary.round(3))
display(null_diagnostics.summary.round(4))

## Appendix E: semantic specificity
Embedding generation is isolated in this cell. If the versioned cache is absent, the cell explicitly issues billed `text-embedding-3-large` calls; all subsequent summaries and figures use only the saved vectors. Set `GENERATE_MISSING_EMBEDDINGS = False` to require a pre-existing cache.

In [ ]:
GENERATE_MISSING_EMBEDDINGS = True
if not EMBEDDING_CACHE.exists():
    if not GENERATE_MISSING_EMBEDDINGS:
        raise FileNotFoundError(f'Missing versioned embedding cache: {EMBEDDING_CACHE}')
    print(f'Generating {len(criteria)} rubric embeddings with {EMBEDDING_MODEL}; this issues billed API calls.')
    generate_embedding_cache(criteria, EMBEDDING_CACHE)
embedding_cache = load_embedding_cache(EMBEDDING_CACHE, criteria)
semantic_diagnostics = semantic_specificity(criteria, embedding_cache)
display(semantic_diagnostics.summary.round(3))
display(semantic_diagnostics.by_level_distance.round(3))
display(semantic_diagnostics.statistics)

## Appendix E: alternative constructions of $\hat y_{ij}$
Fits null-as-zero, complete-case, drop-most-null-class, leave-one-class-out, and single-class-only variants against the current hypothesis set. Per-fit sample sizes are retained in `n`.

In [ ]:
rubric_refits = rubric_sensitivity(
    RESULTS, cells=score_cells, cache_path=OUTPUT / 'rubric_sensitivity.csv'
)
display(sensitivity_summary(rubric_refits).round(3))

In [ ]:
write_appendix_e_outputs(
    OUTPUT, score_cells=score_cells, criteria=criteria,
    consistency=consistency, nulls=null_diagnostics,
    semantics=semantic_diagnostics, sensitivity=rubric_refits,
)
rubric_sensitivity_table = sensitivity_summary(rubric_refits)
rubric_semantics_table_data = rubric_semantics_table(criteria, semantic_diagnostics.pairs)
rubric_semantics_table_data.to_csv(OUTPUT / 'rubric_semantics_table.csv', index=False)
write_rubric_consistency_latex(consistency.summary, OUTPUT / 'tabular_rubric_consistency.tex')
write_rubric_semantics_latex(rubric_semantics_table_data, OUTPUT / 'tabular_rubric_semantics.tex')
write_rubric_sensitivity_latex(rubric_sensitivity_table, OUTPUT / 'tabular_rubric_sensitivity.tex')
write_rubric_sensitivity_latex(rubric_sensitivity_table, OUTPUT / 'tabular_rubric_null.tex')
print('Wrote Appendix E CSVs, LaTeX tables, and Figures E1–E4 to', OUTPUT)